In [2]:
using Revise
using Test
using Statistics
includet("../src/Jackknife.jl")

# Unit Tests
We are going to test the 4 functions in part1: 
```iso_func(n,d)```
```leave_out_func(X,Y,i)```
```ols_est_func(X,Y,i)```
```function covar_func(X,Y)```

Note that Monte Carlo simulations are non-deterministic and therefore can’t be verified using unit tests. We are not going to test the ```error_jk(X,beta_i,sigma,r)```

In [3]:
@testset "Simple Cases" begin
    @testset "Simple Case for leave_out_func" begin 

        X_simple = [1 2;3 4]
        Y_simple = [3;4]
        X_out,Y_out = leave_out_func(X_simple,Y_simple,1)
        @test X_out == [3 4]
        @test Y_out == [4]
    end
    #For the ```ols_est_func```, we make $(X^{T}X)^{-1}X^{T}Y = IY$ and check whether receive the same value Y
    @testset "Simple Case for ols_est__func" begin
        X_simple = [1 0;0 1;2 1]
        Y_simple = [3;4;5]
        beta_out = ols_est_func(X_simple,Y_simple,3)
        @test beta_out == [3;4]
    end
    @testset "Simple Case for covar_func(X,Y)" begin
        n,d = 10,3
        X_simple = randn(n,d)
        Y_simple = randn(n)
        A_simple = covar_func(X_simple,Y_simple)
        @test size(A_simple)==(d,d) 
        @test isapprox(A_simple, A_simple',atol = 1e-10)
    end
    @testset "Simple Case for iso_func(n,d)" begin
        n,d = 100,30
        X_iso = iso_func(n,d)
        result = (1/n)* X_iso' * X_iso
        @test size(X_iso)==(n,d) 
        @test isapprox(result, I, atol = 1e-10)
    end
end

Test Summary: | Pass  Total  Time
Simple Cases  |    7      7  6.2s


Test.DefaultTestSet("Simple Cases", Any[Test.DefaultTestSet("Simple Case for leave_out_func", Any[], 2, false, false, true, 1.772216778091e9, 1.772216778782e9, false, "In[3]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Simple Case for ols_est__func", Any[], 1, false, false, true, 1.772216778782e9, 1.772216779722e9, false, "In[3]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Simple Case for covar_func(X,Y)", Any[], 2, false, false, true, 1.772216779722e9, 1.772216782062e9, false, "In[3]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Simple Case for iso_func(n,d)", Any[], 2, false, false, true, 1.772216782062e9, 1.772216784281e9, false, "In[3]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4

In [4]:
# Corner & Boundary Cases
@testset "Corner & Boundary Cases" begin
    @testset "Boundary leave_out_func(X,Y,i)" begin
        X =  [1 2 3;2 3 4;3 4 5;7 8 9]
        Y = [1.1;2.1;3.1;4.1]
        X_out, Y_out = leave_out_func(X,Y,1)
        @test size(X_out,1) == 3
        @test Y_out == [2.1;3.1;4.1]
    end
    @testset "Boundary leave_out_func(X,Y,i)" begin
        X =  [1 2 3;2 3 4;3 4 5;7 8 9]
        Y = [1.1;2.1;3.1;4.1]
        X_out, Y_out = leave_out_func(X,Y,1)
        @test size(X_out,1) == 3
        @test Y_out == [2.1;3.1;4.1]
        Y_wrong = [1.1;2.1]
        @test_throws AssertionError leave_out_func(X_out,Y_wrong,1)
    end
    
    @testset "Corner covar_func(X,Y)" begin
        # We test whether we can get error when n = 2
        X1 =  [1 2;2 3]
        Y1 = [1.1;2.1]
        X2 = [1 3;2 3;4 5]
        Y2 = [1.2;2.1;1.3]
        @test_throws SingularException covar_func(X1,Y1)
        A_out = covar_func(X2,Y2)
        @test size(A_out) == (2,2)
        @test !any(isinf.(A_out) ) 
    end
    @testset "Corner iso_func(n,d)" begin
        # We test the case when n = d =3, the minimum condition of full rank
        n,d =3,3
        X_iso_corner = iso_func(n,d)
        cov_matrix = (1/n)*X_iso_corner' *X_iso_corner
        @test isapprox(cov_matrix, I, atol = 1e-10)
    end

    # 4. ols_est_func 
    @testset "Corner & Bug: ols_est_func" begin
        # Corner Case：Zero Degrees of Freedom
        # When we remove a row, n-1 =d, X_out is a square matrix
        X_perfect = [1 2;3 4; 9.9 9.9] # n=3, d=2
        Y_perfect = [10.0, 20.0, 99.0]
        beta_corner = ols_est_func(X_perfect, Y_perfect, 3)
        @test length(beta_corner) == 2
        # [1.0 2.0; 3.0 4.0] \ [10.0, 20.0] = [0.0, 5.0]
        @test isapprox(beta_corner,[0.0, 5.0],atol = 1e-10)
    end
end

Test Summary:           | Pass  Total  Time
Corner & Boundary Cases |   11     11  1.4s


Test.DefaultTestSet("Corner & Boundary Cases", Any[Test.DefaultTestSet("Boundary leave_out_func(X,Y,i)", Any[], 2, false, false, true, 1.77221678633e9, 1.772216786526e9, false, "In[4]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Boundary leave_out_func(X,Y,i)", Any[], 3, false, false, true, 1.772216786526e9, 1.772216786776e9, false, "In[4]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Corner covar_func(X,Y)", Any[], 3, false, false, true, 1.772216786776e9, 1.772216787437e9, false, "In[4]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Corner iso_func(n,d)", Any[], 1, false, false, true, 1.772216787437e9, 1.772216787438e9, false, "In[4]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5

In [5]:
#Bug catch
@testset "Bug Catch Tests" begin
    #  leave_out_func 
    @testset "Bug: leave_out_func" begin
        X_test = rand(5, 2)
        Y_test = rand(5)
        Y_wrong = rand(3) # Length doesn't match
        @test_throws AssertionError leave_out_func(X_test, Y_wrong, 1)
    
        # Remove row/column that does not exist
        @test_throws AssertionError leave_out_func(X_test, Y_test, 6)
        @test_throws AssertionError leave_out_func(X_test, Y_test, 0)
    end

    # 2. ols_est_func 
    @testset "Bug: ols_est_func" begin
        X_test = rand(5, 2)
        Y_test = rand(5)
        @test_throws AssertionError ols_est_func(X_test, Y_test, 6)
    end
    # 3. covar_func 
    @testset "Bug: covar_func" begin
        X_test = rand(5, 2)
        Y_wrong = rand(3)
        # Length Does Not Match
        @test_throws AssertionError covar_func(X_test, Y_wrong)
        
        # degree of freedom is not enough
        X_sing = [1.0 2.0; 3.0 4.0]
        Y_sing = [5.0, 6.0]
        @test_throws SingularException covar_func(X_sing, Y_sing)
    end

    # 4. iso_func 
    @testset "Bug: iso_func" begin
        # n < d, will cause Inf 
        n, d = 2, 5
        @test_throws DomainError iso_func(n, d)
    end
end


Test Summary:   | Pass  Total  Time
Bug Catch Tests |    7      7  0.1s


Test.DefaultTestSet("Bug Catch Tests", Any[Test.DefaultTestSet("Bug: leave_out_func", Any[], 3, false, false, true, 1.77221678919e9, 1.772216789229e9, false, "In[5]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Bug: ols_est_func", Any[], 1, false, false, true, 1.772216789229e9, 1.772216789229e9, false, "In[5]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Bug: covar_func", Any[], 2, false, false, true, 1.772216789229e9, 1.772216789268e9, false, "In[5]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9)), Test.DefaultTestSet("Bug: iso_func", Any[], 1, false, false, true, 1.772216789268e9, 1.772216789268e9, false, "In[5]", Random.Xoshiro(0xbdc79bc841315e7a, 0x631f3b8126638dbc, 0x8182f37b4aa5cab5, 0xa5efae772591233e, 0xf841d4f63090f3e9))], 0